Importing libraries and initialising global constants. This code assumes wavefunctions of the from Φ_i(r1)Φ_j(r2) - Φ_j(r1)Φ_i(r2), with l=0 in the GEM. All matrix elements should be treated with care. For the normalisation choice when i=j see mathematica notebook for explanation.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.special import gamma, factorial

STARTING_RANGE_PARAMETER = 1 # In [fm^-2]
ENDING_RANGE_PARAMETER = 15
REDUCED_MASS = 935 * 10/11 # In [Mev / c^2], need to update value and units (10/11 A in MeV)
SUM_LIMIT = 10 # Determines the number of gaussians we expand our wave function to
CORE_MS_RADIUS = 2.30**2 # In fm^2, taken from p.232 of Tanihata et. al. (2013)

V_LS = 21.0 # In MeV
DIFFUSIVITY = 0.6 # Diffusivity, may want to check the vaidity of this paticular number
r_0 = 1.2 # In fm, may want to chose a better value for small nuclei
A_C = 10 # The number of nucleons in the core
SIGMA = 2 #fm
TWO_PARTICLE_POTENTIAL_DEPTH = -24.22 * 0 # MeV

CENTRAL_POTENTIAL_PARAMETERS = [0.1, 0.151991, 0.231013, 0.351119, 0.53367, 0.811131, 1.23285,
                                1.87382, 2.84804, 4.32876, 6.57933, 10.]

CENTRAL_MIXING_COEFFICIENTS = [0.0558247,0.214443,2.42773,-0.724055,-2.17761,1.02031,0.819031,-0.96538,0.197094,0.3221,-0.296652,0.093208]

In [10]:
def normalisation(range_parameter_i, range_parameter_j):
    return 1 / np.sqrt(2 * (1 - single_particle_overlap(range_parameter_i, range_parameter_j)))
    
def single_particle_overlap(range_parameter_i, range_parameter_j):
    return ((2 * range_parameter_i * range_parameter_j) / (range_parameter_i**2 + range_parameter_j**2))**(1.5)

def single_particle_potential_element(range_parameter_i, range_parameter_j, central_potential_mixing_coefficient,
                             central_potential_param):
    V_0 = -11.405 * (-1)**0 - 51.175 # Defines V_0 for odd and even l states, shifted from values in capel et. al.
    term_1 = 2 / (range_parameter_i * range_parameter_j)
    term_2 = 1 / ((1 / range_parameter_i**2) + (1 / range_parameter_j**2) + central_potential_param)
    return (term_1 * term_2)**(1.5)

def single_particle_kinetic_element(range_parameter_i, range_parameter_j, μ=REDUCED_MASS):
    term_1 = range_parameter_i**2 + range_parameter_j**2
    return (197**2 / (2 * μ)) * 6 * single_particle_overlap(range_parameter_i, range_parameter_j) / term_1

def overlap_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l):
    if range_parameter_i == range_parameter_j or range_parameter_k == range_parameter_l:
        mat_elem = 0
    else:
        term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_overlap(range_parameter_j, range_parameter_l)
        term_2 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_overlap(range_parameter_j, range_parameter_k)
        term_3 = normalisation(range_parameter_i, range_parameter_j) * normalisation(range_parameter_k, range_parameter_l)
        mat_elem = 2 * (term_1 - term_2) / term_3

    return mat_elem

def kinetic_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, μ=REDUCED_MASS, orb_ang_momentum=0):
    if range_parameter_i == range_parameter_j or range_parameter_k == range_parameter_l:
        mat_elem = 0
    else:
        term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_kinetic_element(
            range_parameter_j, range_parameter_l)
        term_2 = single_particle_overlap(range_parameter_j, range_parameter_l) * single_particle_kinetic_element(
            range_parameter_i, range_parameter_k)
        term_3 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_kinetic_element(
            range_parameter_j, range_parameter_k)
        term_4 = single_particle_overlap(range_parameter_j, range_parameter_k) * single_particle_kinetic_element(
            range_parameter_i, range_parameter_l)
        term_5 = normalisation(range_parameter_i, range_parameter_j) * normalisation(range_parameter_k, range_parameter_l)

        mat_elem = (term_1 + term_2 - term_3 - term_4) / term_5
    return mat_elem

def potential_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, central_potential_mixing_coefficient,
                             central_potential_param):
    V_0 = -11.405 * (-1)**0 - 51.175 # Defines V_0 for odd and even l states, shifted from values in capel et. al.
    if range_parameter_i == range_parameter_j or range_parameter_k == range_parameter_l:
        mat_elem = 0
    else:
        term_1 = single_particle_overlap(range_parameter_i, range_parameter_k) * single_particle_potential_element(
            range_parameter_j, range_parameter_l, central_potential_mixing_coefficient, central_potential_param)
        term_2 = single_particle_overlap(range_parameter_j, range_parameter_l) * single_particle_potential_element(
            range_parameter_i, range_parameter_k, central_potential_mixing_coefficient, central_potential_param)
        term_3 = single_particle_overlap(range_parameter_i, range_parameter_l) * single_particle_potential_element(
            range_parameter_j, range_parameter_k, central_potential_mixing_coefficient, central_potential_param)
        term_4 = single_particle_overlap(range_parameter_j, range_parameter_k) * single_particle_potential_element(
            range_parameter_i, range_parameter_l, central_potential_mixing_coefficient, central_potential_param)
        term_5 = normalisation(range_parameter_i, range_parameter_j) * normalisation(range_parameter_k, range_parameter_l)

        mat_elem = (term_1 + term_2 - term_3 - term_4) / term_5
    return mat_elem

def interaction_matrix_element(range_parameter_i, range_parameter_j, range_parameter_k, range_parameter_l, potential_parameter,
                              potential_strength=TWO_PARTICLE_POTENTIAL_DEPTH):
    def ith_term(param_1, param_2, param_3, param_4, potential_param):
        term_A = param_1**(-2) + param_2**(-2) + potential_parameter**(-2)
        term_B = param_3**(-2) + param_4**(-2) + (param_1**2 + param_2**2) / ((
            param_2 * potential_parameter)**2 * (1 + param_1**2))
        return (term_A * term_B)**(-1.5)
    if range_parameter_i == range_parameter_j or range_parameter_k == range_parameter_l:
        mat_elem = 0
    else:
        term_1 = ith_term(range_parameter_j, range_parameter_l, range_parameter_i, range_parameter_k, potential_parameter)
        term_2 = ith_term(range_parameter_i, range_parameter_l, range_parameter_j, range_parameter_k, potential_parameter)
        mat_elem = (1 / 8) * np.pi * potential_strength * (term_1 - term_2) * normalisation(
            range_parameter_i, range_parameter_j) * normalisation(range_parameter_k, range_parameter_l)
    return mat_elem

In [11]:
#ORDER OF VARIABLES IN MAT ELEM HAS CHANGED!!!
def two_particle_basis_matrix_generation(central_mixing_coefficients=CENTRAL_MIXING_COEFFICIENTS,
                                         central_potential_parameters=CENTRAL_POTENTIAL_PARAMETERS,
                                         size=SUM_LIMIT, potential_parameter=SIGMA):

    first_h_matrix = np.zeros(shape=(size**2, size**2))
    second_h_matrix = np.zeros(shape=(size**2, size**2))
    interaction_matrix = np.zeros(shape=(size**2, size**2))
    overlap_matrix = np.zeros(shape=(size**2, size**2))

    for I in range(1, size**2 + 1):
        i = np.mod((I - 1), size) + 1
        i_range_parameter = next_range_parameter(i - 1)
        j = np.floor((I - 1) / size) + 1
        j_range_parameter = next_range_parameter(j - 1)
        for J in range(1, size**2 + 1):
            k = np.mod((J - 1), size) + 1
            k_range_parameter = next_range_parameter(k - 1)
            l = np.floor((J - 1) / size) + 1
            l_range_parameter = next_range_parameter(l - 1)

            overlap_matrix[I - 1, J - 1] = overlap_matrix_element(i_range_parameter, j_range_parameter,
                                                                  k_range_parameter, l_range_parameter)

            interaction_matrix[I - 1, J - 1] = interaction_matrix_element(i_range_parameter, j_range_parameter,
                                                                          k_range_parameter, l_range_parameter,
                                                                          potential_parameter)

            first_potential_energy_term = 0
            first_kinetic_term = kinetic_matrix_element(i_range_parameter, j_range_parameter,
                                                        k_range_parameter, l_range_parameter)
            for m in range(len(central_mixing_coefficients)):
                first_potential_energy_term += potential_matrix_element(
                    i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                    central_mixing_coefficients[m], central_potential_parameters[m])
            first_h_matrix[I - 1, J - 1] = (first_kinetic_term + first_potential_energy_term)

            second_potential_energy_term = 0
            second_kinetic_term = kinetic_matrix_element(i_range_parameter, j_range_parameter,
                                                         k_range_parameter, l_range_parameter)
            for m in range(len(central_mixing_coefficients)):
                second_potential_energy_term += potential_matrix_element(
                    i_range_parameter, j_range_parameter, k_range_parameter, l_range_parameter,
                    central_mixing_coefficients[m], central_potential_parameters[m])
            second_h_matrix[I - 1, J - 1] = (second_kinetic_term + second_potential_energy_term)

    np.savetxt('nmatrix.csv', overlap_matrix)
    np.savetxt('h1matrix.csv', first_h_matrix)
    np.savetxt('h2matrix.csv', second_h_matrix)
    np.savetxt('interaction.csv', interaction_matrix)
    
    return first_h_matrix + second_h_matrix + interaction_matrix, overlap_matrix

def next_range_parameter(i, starting_range_parameter=STARTING_RANGE_PARAMETER, ending_range_parameter=ENDING_RANGE_PARAMETER,
                         sum_limit=SUM_LIMIT):
    """
    Finds the next range parameter given the previous and initial range parameters.
    Currently using a simple geometric series to determine range parameters.
    Chose geometric basis parameters $\alpha_i = \alpha_1a^{i-1}$ with initial parameters $\alpha_1 = 0.01, a=2$

    Parameters
    ----------
    i : int detailing the iteration number

    Returns
    -------
    new_range_parameter: float

    """
    geometric_progression_number = (ending_range_parameter / starting_range_parameter)**(1 / (sum_limit - 1))
    new_range_parameter = starting_range_parameter * geometric_progression_number**(i)

    return new_range_parameter
h_matrix, n_matrix = two_particle_basis_matrix_generation()

In [12]:
s_eigenvalues, s_eigenvectors = scipy.linalg.eig(h_matrix, n_matrix)
#bound_states = np.real(s_eigenvalues) < 0
s_overlap_eigenvalues, s_overlap_eigenvectors = scipy.linalg.eigh(n_matrix)
s_overlap_matrix_condition_number = np.max(s_overlap_eigenvalues) / np.min(s_overlap_eigenvalues)
print(f"The s 1/2 overlap matrix condition number is", s_overlap_matrix_condition_number)

s0_eigenvector = np.asmatrix(s_eigenvectors[:, 0])
s1_eigenvector = np.asmatrix(s_eigenvectors[:, 1])
print("The S state eigenvalues are", s_eigenvalues)
#print(f'The bound states have energies {bound_states}')
#print("The S1 eigenvector is", s1_eigenvector)
print(f'The bound states have energies{np.real(np.sort(s_eigenvalues[np.real(s_eigenvalues) < 0]))}')

The s 1/2 overlap matrix condition number is -2.6002646324438364e+16
The S state eigenvalues are [            nan       +nanj             nan       +nanj
             nan       +nanj             nan       +nanj
             nan       +nanj             nan       +nanj
             nan       +nanj             nan       +nanj
             nan       +nanj  9.14320218e+01+0.j
             inf+0.j         -5.99506118e+00+0.j
             inf+0.j                     inf+0.j
             inf+0.j                     inf+0.j
             inf+0.j          1.11074840e+06+0.j
 -2.68746391e+06+0.j          8.31278237e+04+0.j
 -5.05558804e+05+0.j         -1.21604377e+05+0.j
 -1.03385971e+06+0.j          2.27303853e+05+0.j
  2.73228435e+05+0.j          8.87726371e+04+0.j
  1.19507990e+05+0.j          5.65213976e+04+0.j
  7.24866619e+04+0.j          2.89001851e+04+0.j
 -3.20224241e+04+0.j          9.82304875e+04+0.j
  1.72615720e+04+0.j          1.49595622e+03+0.j
 -6.95095257e+04+0.j          2.019188